In [11]:
import os

# 인증 설정
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# 데이터 저장 폴더 생성
os.makedirs('/content/data', exist_ok=True)

# 데이터셋 다운로드
!kaggle competitions download -c ai09-level1-project -p /content/data

# 압축 해제
!unzip /content/data/ai09-level1-project.zip -d /content/data/

100% 1.79G/1.79G [00:27<00:00, 70.7MB/s]

Archive:  /content/data/ai09-level1-project.zip
  inflating: /content/data/sprint_ai_project1_data/test_images/1.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/10.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/100.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1003.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1004.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1005.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1006.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1007.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1009.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1010.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1012.png  
  inflating: /content/data/sprint_ai_project1_data/test_images/1013.png  
  inflating: /content/data/s

In [4]:
import torch

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [12]:
import os
import glob

# 경로 설정
base_dir = '/content/data/sprint_ai_project1_data'

train_img_dir = os.path.join(base_dir, 'train_images')       # 학습 이미지
train_json_dir = os.path.join(base_dir, 'train_annotations') # 학습 annotation
test_img_dir = os.path.join(base_dir, 'test_images')         # 테스트 이미지

# 파일 목록 가져오기
train_img_files = glob.glob(os.path.join(train_img_dir, "*.png"))
test_img_files = glob.glob(os.path.join(test_img_dir, "*.png"))
train_json_files = glob.glob(os.path.join(train_json_dir, "**/*.json"), recursive=True)

print(f"학습 이미지 수: {len(train_img_files)}")
print(f"테스트 이미지 수: {len(test_img_files)}")
print(f"Annotation 수: {len(train_json_files)}")

학습 이미지 수: 232
테스트 이미지 수: 842
Annotation 수: 763


**① 리사이즈 방식: 단순 resize → Letterbox**  
기존에는 976×1280을 단순히 늘리거나 줄였겠지만,   
YOLO11은 비율을 유지한 채 회색(114) 패딩을 채웁니다.   
약처럼 형태 왜곡에 민감한 이미지는 특히 중요합니다.  
**② 입력 크기: 976×1280 → 640×640**  
YOLO11의 권장 입력 크기는 640×640입니다.  
기존 크기를 그대로 넣으면 메모리와 속도에서 크게 불리해요.  
**③ 디렉토리 구조: 라벨 파일 → 폴더 구조로 클래스 표현**  
YOLO11 Classification은 별도 라벨 파일 없이 폴더 이름 = 클래스명 방식을 사용합니다.   
COCO JSON을 파싱해서 이 구조로 자동 변환해줍니다.

In [16]:
"""
YOLO11 Classification 전처리 파이프라인
- 기존: 커스텀 CNN (976×1280, COCO JSON 라벨)
- 변환: YOLO11 Classification 호환 포맷

YOLO11 Classification 디렉토리 구조:
    dataset/
    ├── train/
    │   ├── class_A/
    │   │   ├── img1.jpg
    │   │   └── img2.jpg
    │   └── class_B/
    │       └── img3.jpg
    ├── val/
    │   ├── class_A/
    │   └── class_B/
    └── test/          (선택)
        ├── class_A/
        └── class_B/
"""

'\nYOLO11 Classification 전처리 파이프라인\n- 기존: 커스텀 CNN (976×1280, COCO JSON 라벨)\n- 변환: YOLO11 Classification 호환 포맷\n \nYOLO11 Classification 디렉토리 구조:\n    dataset/\n    ├── train/\n    │   ├── class_A/\n    │   │   ├── img1.jpg\n    │   │   └── img2.jpg\n    │   └── class_B/\n    │       └── img3.jpg\n    ├── val/\n    │   ├── class_A/\n    │   └── class_B/\n    └── test/          (선택)\n        ├── class_A/\n        └── class_B/\n'

In [1]:
!pip install ultralytics albumentations opencv-python Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00


In [14]:
import os
import json
import shutil
import random
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [15]:
# ──────────────────────────────────────────────
# 0. 설정값
# ──────────────────────────────────────────────
class Config:
    ANNO_DIR    = train_json_dir
    IMAGE_DIR   = train_img_dir
    OUTPUT_DIR  = "/content/data/dataset_yolo11"

    IMG_SIZE    = 640          # YOLO11 권장 입력 크기
    BBOX_MARGIN = 0.1          # bbox crop 시 여백 비율 (10%)

    SPLIT_RATIO = (0.8, 0.1, 0.1)   # train : val : test
    SEED        = 42

    # 클래스명 대신 약품 코드(K-XXXXXX)를 폴더명으로 쓸지 여부
    # True  → 폴더명: "K-018147"         (짧고 안전)
    # False → 폴더명: "리리카캡슐 150mg" (가독성 좋지만 특수문자 주의)
    USE_CODE_AS_CLASSNAME = False

cfg = Config()
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)

In [17]:
# ──────────────────────────────────────────────
# 1. 모든 JSON 파싱 → crop 단위 레코드 수집
# ──────────────────────────────────────────────
def collect_records(anno_dir: str) -> list[dict]:
    """
    반환값: [
      {
        "image_path": "/content/data/.../K-003351-016688-018147_0_2_0_2_70_000_200.png",
        "bbox": [x, y, w, h],        ← COCO 형식 (pixel)
        "class_code": "K-018147",
        "class_name": "리리카캡슐 150mg",
        "json_path": "..."
      }, ...
    ]
    """
    records = []
    missing_images = []

    anno_root = Path(anno_dir)
    json_files = list(anno_root.rglob("*.json"))
    print(f"총 JSON 파일 수: {len(json_files)}")

    for json_path in json_files:
        # 알약 코드는 JSON의 부모 폴더명 (예: K-018147)
        class_code = json_path.parent.name

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # 이미지 파일명
        file_name = data["images"][0]["file_name"]
        image_path = Path(cfg.IMAGE_DIR) / file_name

        if not image_path.exists():
            missing_images.append(str(image_path))
            continue

        # 해당 클래스(class_code)의 annotation만 추출
        # category_id = 숫자 (예: 18147) → K-XXXXXX 변환
        target_cat_id = None
        for cat in data["categories"]:
            if f"K-{cat['id']}" == class_code:
                target_cat_id = cat["id"]
                class_name = cat["name"]
                break

        if target_cat_id is None:
            # category_id 직접 매핑 실패 시 폴더명으로 추론
            # K-018147 → 18147
            try:
                code_num = int(class_code.replace("K-", ""))
                for cat in data["categories"]:
                    if cat["id"] == code_num:
                        target_cat_id = cat["id"]
                        class_name = cat["name"]
                        break
            except ValueError:
                continue

        if target_cat_id is None:
            continue

        # 해당 category_id의 bbox 추출
        for ann in data["annotations"]:
            if ann["category_id"] == target_cat_id:
                records.append({
                    "image_path": str(image_path),
                    "bbox": ann["bbox"],          # [x, y, w, h]
                    "class_code": class_code,
                    "class_name": class_name,
                    "json_path": str(json_path),
                })
                break  # 1 JSON = 1 알약 bbox

    if missing_images:
        print(f"[경고] 이미지 없음: {len(missing_images)}건")

    # 클래스 분포 출력
    class_counts = defaultdict(int)
    for r in records:
        class_counts[r["class_code"]] += 1
    print(f"\n총 crop 레코드: {len(records)}개 | 클래스 수: {len(class_counts)}")
    for code, cnt in sorted(class_counts.items()):
        name = next(r["class_name"] for r in records if r["class_code"] == code)
        print(f"  {code} ({name}): {cnt}개")

    return records

In [18]:
# ──────────────────────────────────────────────
# 2. Stratified Split
# ──────────────────────────────────────────────
def split_records(records: list[dict], split_ratio: tuple) -> dict:
    """클래스별 균등 분할"""
    class2records = defaultdict(list)
    for r in records:
        class2records[r["class_code"]].append(r)

    splits = {"train": [], "val": [], "test": []}
    train_r, val_r, _ = split_ratio

    for code, recs in class2records.items():
        random.shuffle(recs)
        n = len(recs)
        n_train = max(1, int(n * train_r))
        n_val   = max(1, int(n * val_r)) if n > 2 else 0

        splits["train"].extend(recs[:n_train])
        splits["val"].extend(recs[n_train:n_train + n_val])
        splits["test"].extend(recs[n_train + n_val:])

    for split, recs in splits.items():
        print(f"  [{split:5s}] {len(recs)}개")

    return splits

In [19]:
# ──────────────────────────────────────────────
# 3. bbox crop + Letterbox 리사이즈
# ──────────────────────────────────────────────
def crop_and_letterbox(image: np.ndarray, bbox: list, margin: float, target_size: int) -> np.ndarray | None:
    """
    1) bbox 영역을 margin 포함해서 crop
    2) Letterbox 리사이즈 (비율 유지 + 회색 패딩)
    """
    h, w = image.shape[:2]
    x, y, bw, bh = bbox

    # margin 적용
    mx = int(bw * margin)
    my = int(bh * margin)
    x1 = max(0, int(x) - mx)
    y1 = max(0, int(y) - my)
    x2 = min(w, int(x + bw) + mx)
    y2 = min(h, int(y + bh) + my)

    if x2 <= x1 or y2 <= y1:
        return None

    cropped = image[y1:y2, x1:x2]

    # Letterbox
    ch, cw = cropped.shape[:2]
    scale = target_size / max(ch, cw)
    new_w, new_h = int(cw * scale), int(ch * scale)
    resized = cv2.resize(cropped, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.full((target_size, target_size, 3), 114, dtype=np.uint8)
    pad_top  = (target_size - new_h) // 2
    pad_left = (target_size - new_w) // 2
    canvas[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized

    return canvas

In [20]:
# ──────────────────────────────────────────────
# 4. Augmentation 파이프라인
# ──────────────────────────────────────────────
def get_train_transform() -> A.Compose:
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, value=114, p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    ])

In [21]:
# ──────────────────────────────────────────────
# 5. 전처리 실행 및 저장
# ──────────────────────────────────────────────
def preprocess_and_save(splits: dict):
    output_root = Path(cfg.OUTPUT_DIR)
    train_transform = get_train_transform()

    for split, records in splits.items():
        is_train = (split == "train")
        saved, skipped = 0, 0

        for rec in records:
            # 클래스 폴더명 결정
            folder_name = rec["class_code"] if cfg.USE_CODE_AS_CLASSNAME else rec["class_name"]
            # Windows/Linux 안전 문자로 치환
            folder_name = folder_name.replace("/", "_").replace("\\", "_")

            save_dir = output_root / split / folder_name
            save_dir.mkdir(parents=True, exist_ok=True)

            # 이미지 로드
            img_bgr = cv2.imread(rec["image_path"])
            if img_bgr is None:
                skipped += 1
                continue
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            # crop + letterbox
            img_crop = crop_and_letterbox(img_rgb, rec["bbox"], cfg.BBOX_MARGIN, cfg.IMG_SIZE)
            if img_crop is None:
                skipped += 1
                continue

            # train만 augmentation 적용 (저장용이므로 정규화 제외)
            if is_train:
                img_crop = train_transform(image=img_crop)["image"]

            # 저장 파일명: {이미지 stem}_{class_code}.jpg
            img_stem = Path(rec["image_path"]).stem
            save_path = save_dir / f"{img_stem}_{rec['class_code']}.jpg"
            cv2.imwrite(str(save_path), cv2.cvtColor(img_crop, cv2.COLOR_RGB2BGR))
            saved += 1

        print(f"  [{split:5s}] 저장 {saved}개 | 스킵 {skipped}개")

    print(f"\n[완료] 출력 경로: {output_root}")

In [22]:
# ──────────────────────────────────────────────
# 6. dataset.yaml 생성
# ──────────────────────────────────────────────
def generate_yaml(records: list[dict]):
    if cfg.USE_CODE_AS_CLASSNAME:
        classes = sorted({r["class_code"] for r in records})
    else:
        classes = sorted({r["class_name"].replace("/", "_") for r in records})

    yaml_path = Path(cfg.OUTPUT_DIR) / "dataset.yaml"
    content = f"""# YOLO11 Classification Dataset
path: {Path(cfg.OUTPUT_DIR).resolve()}
train: train
val: val
test: test

nc: {len(classes)}
names: {classes}
"""
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"[YAML] {yaml_path}")